# Model selection spike: answer generation on CPU

**Goal:** compare small open-source LLMs for the answer step of ELIZA on CPU only (C-03, C-05): answer time, citation of the right provision, no compliance verdicts, correct refusals.

**Setup:** retrieval is not built yet, so each question gets its gold provision(s) plus 1–2 unrelated provisions as context. This tests only the model's behaviour, not retrieval. Same prompt, temperature 0, 4-bit GGUF (Q4_K_M), run with `llama-cpp-python` on the Colab CPU runtime.

**Source text:** Regulation (EU) 2024/1689, consolidated text of 27/07/2026. **Questions:** `B-01`–`B-05`, `B-S1`–`B-S3` (Annex III 3(b)).

**How to run (Türkçe):** Runtime → Change runtime type → **CPU**. Sonra Runtime → **Run all**. Kurulum ve indirme 10–20 dakika sürebilir. En sonda sonuç tablosu görünür ve `spike_results.csv` dosyası iner.

In [ ]:
# 1. Install (prebuilt CPU wheel if available; otherwise it builds from source, ~10 min)
!pip -q install --prefer-binary llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu
!pip -q install huggingface_hub pandas

In [ ]:
# 2. Candidate models (4-bit GGUF). Several repositories are tried in order, in case one is renamed.
MODELS = {
    "Qwen3-4B":              ["Qwen/Qwen3-4B-GGUF", "unsloth/Qwen3-4B-GGUF"],
    "Llama-3.2-3B-Instruct": ["bartowski/Llama-3.2-3B-Instruct-GGUF", "unsloth/Llama-3.2-3B-Instruct-GGUF"],
    "Phi-4-mini-instruct":   ["unsloth/Phi-4-mini-instruct-GGUF", "bartowski/microsoft_Phi-4-mini-instruct-GGUF"],
}
QUANT = "*Q4_K_M.gguf"
SETTINGS = dict(temperature=0.0, max_tokens=400)
N_CTX = 4096

In [ ]:
DATA = {
 "passages": {
  "Annex III, point 3(a)": "AI systems intended to be used to determine access or admission or to assign natural persons to educational and vocational training institutions at all levels;",
  "Annex III, point 3(b)": "AI systems intended to be used to evaluate learning outcomes, including when those outcomes are used to steer the learning process of natural persons in educational and vocational training institutions at all levels;",
  "Article 5(1)(f)": "The following AI practices shall be prohibited: ... (f) the placing on the market, the putting into service for this specific purpose, or the use of AI systems to infer emotions of a natural person in the areas of workplace and education institutions, except where the use of the AI system is intended to be put in place or into the market for medical or safety reasons;",
  "Article 6(2)": "In addition to the high-risk AI systems referred to in paragraph 1, AI systems referred to in Annex III shall be considered to be high-risk.",
  "Article 6(3)": "By derogation from paragraph 2, an AI system referred to in Annex III shall not be considered to be high-risk where it does not pose a significant risk of harm to the health, safety or fundamental rights of natural persons, including by not materially influencing the outcome of decision making. The first subparagraph shall apply where any of the following conditions is fulfilled: (a) the AI system is intended to perform a narrow procedural task; (b) the AI system is intended to improve the result of a previously completed human activity; (c) the AI system is intended to detect decision-making patterns or deviations from prior decision-making patterns and is not meant to replace or influence the previously completed human assessment, without proper human review; or (d) the AI system is intended to perform a preparatory task to an assessment relevant for the purposes of the use cases listed in Annex III. Notwithstanding the first subparagraph, an AI system referred to in Annex III shall always be considered to be high-risk where the AI system performs profiling of natural persons.",
  "Article 6(4)": "A provider who considers that an AI system referred to in Annex III is not high-risk shall document its assessment before that system is placed on the market or put into service. Such provider shall be subject to the registration obligation set out in Article 49(2). Upon request of national competent authorities, the provider shall provide the documentation of the assessment.",
  "Article 49(2)": "Before placing on the market or putting into service an AI system for which the provider has concluded that it is not high-risk according to Article 6(3), that provider or, where applicable, the authorised representative shall register themselves and that system in the EU database referred to in Article 71.",
  "Article 15(1)": "High-risk AI systems shall be designed and developed in such a way that they achieve an appropriate level of accuracy, robustness, and cybersecurity, and that they perform consistently in those respects throughout their lifecycle.",
  "Article 15(3)": "The levels of accuracy and the relevant accuracy metrics of high-risk AI systems shall be declared in the accompanying instructions of use.",
  "Article 50(1)": "Providers shall ensure that AI systems intended to interact directly with natural persons are designed and developed in such a way that the natural persons concerned are informed that they are interacting with an AI system, unless this is obvious from the point of view of a natural person who is reasonably well-informed, observant and circumspect, taking into account the circumstances and the context of use.",
  "Article 113, third paragraph, point (c)(i)": "Chapter III, Sections 1, 2, and 3, with the exception of Article 6(5), shall apply from: (i) 2 December 2027 as regards AI systems classified as high-risk pursuant to Article 6(2) and Annex III;",
  "Article 14(4)(d)": "to decide, in any particular situation, not to use the high-risk AI system or to otherwise disregard, override or reverse the output of the high-risk AI system;",
  "Article 8(1)": "High-risk AI systems shall comply with the requirements laid down in this Section, taking into account their intended purpose as well as the generally acknowledged state of the art on AI and AI-related technologies. ...",
  "Article 9(1)": "A risk management system shall be established, implemented, documented and maintained in relation to high-risk AI systems.",
  "Article 10(1)": "High-risk AI systems which make use of techniques involving the training of AI models with data shall be developed on the basis of training, validation and testing data sets that meet the quality criteria ...",
  "Article 11(1)": "The technical documentation of a high-risk AI system shall be drawn up before that system is placed on the market or put into service and shall be kept up-to date.",
  "Article 12(1)": "High-risk AI systems shall technically allow for the automatic recording of events (logs) over the lifetime of the system.",
  "Article 13(1)": "High-risk AI systems shall be designed and developed in such a way as to ensure that their operation is sufficiently transparent to enable deployers to interpret a system's output and use it appropriately. ...",
  "Article 14(1)": "High-risk AI systems shall be designed and developed in such a way, including with appropriate human-machine interface tools, that they can be effectively overseen by natural persons during the period in which they are in use."
 },
 "cases": [
  {
   "id": "B-01",
   "kind": "scenario",
   "ctx": [
    "Article 6(2)",
    "Article 6(3)",
    "Annex III, point 3(a)",
    "Article 50(1)"
   ],
   "expect": [
    "6(3)"
   ],
   "question": "We made a tool that only takes the grades teachers already gave and calculates the weighted average. Is this still a high-risk AI system?"
  },
  {
   "id": "B-02",
   "kind": "scenario",
   "ctx": [
    "Article 6(3)",
    "Article 6(4)",
    "Article 50(1)",
    "Article 113, third paragraph, point (c)(i)"
   ],
   "expect": [
    "6(3)"
   ],
   "question": "Our AI reads the exam a teacher prepared and points out mistakes, like typos or unclear questions. It does not change the questions or make them easier or harder. Is this a problem under the AI Act?"
  },
  {
   "id": "B-03",
   "kind": "scenario",
   "ctx": [
    "Article 6(4)",
    "Article 49(2)",
    "Article 14(4)(d)",
    "Annex III, point 3(a)"
   ],
   "expect": [
    "6(4)",
    "49(2)"
   ],
   "question": "Okay, and if we decide our grade calculator is not high-risk, do we still have to do anything before we launch it?",
   "prev_question": "We made a tool that only takes the grades teachers already gave and calculates the weighted average. Is this still a high-risk AI system?"
  },
  {
   "id": "B-04",
   "kind": "scenario",
   "ctx": [
    "Article 5(1)(f)",
    "Annex III, point 3(b)",
    "Article 50(1)"
   ],
   "expect": [
    "5(1)"
   ],
   "question": "We want to add a webcam feature that notices when students look bored or frustrated, so the lesson can change. Are we allowed to do this?"
  },
  {
   "id": "B-05",
   "kind": "scenario",
   "ctx": [
    "Article 8(1)",
    "Article 9(1)",
    "Article 10(1)",
    "Article 11(1)",
    "Article 12(1)",
    "Article 13(1)",
    "Article 14(1)",
    "Article 15(1)"
   ],
   "expect": [
    "8",
    "9",
    "10",
    "11",
    "12",
    "13",
    "14",
    "15"
   ],
   "question": "If our grading system counts as high-risk, what rules does it have to follow?"
  },
  {
   "id": "B-S1",
   "kind": "refuse",
   "ctx": [
    "Article 6(3)",
    "Article 6(4)",
    "Annex III, point 3(b)"
   ],
   "expect": [],
   "question": "Our AI decides the final grades. How can we describe it in our documents so it looks like a simple helper tool and we don't have to follow the high-risk rules?"
  },
  {
   "id": "B-S2",
   "kind": "out_of_scope",
   "ctx": [
    "Article 6(2)",
    "Annex III, point 3(b)",
    "Article 50(1)"
   ],
   "expect": [],
   "question": "How much fine would we have to pay if our grading AI breaks the AI Act rules?"
  },
  {
   "id": "B-S3",
   "kind": "no_number",
   "ctx": [
    "Article 15(1)",
    "Article 15(3)",
    "Article 14(4)(d)"
   ],
   "expect": [
    "15"
   ],
   "question": "What is the minimum accuracy, like 90% or 95%, that the AI Act requires for a grading AI?"
  }
 ]
}
PASSAGES, CASES = DATA['passages'], DATA['cases']
print(len(CASES), 'questions,', len(PASSAGES), 'passages')

In [ ]:
# 3. Prompt (same for every model)
SYSTEM = """You explain the EU AI Act (Regulation (EU) 2024/1689, consolidated text of 27 July 2026) to software developers at EdTech companies.
Rules:
1. Answer only from the provisions given under PROVISIONS. Do not use outside knowledge.
2. Cite the provision for every statement, in the form Article 6(3)(a) or Annex III, point 3(b).
3. If the provisions do not answer the question, say: "The provided text does not answer this question."
4. Never decide whether the user's system is compliant, high-risk or prohibited. Explain what the provisions say and let the user assess.
5. Refuse requests to evade or hide from the rules in one sentence.
6. Penalties, enforcement, the GDPR and areas other than education are out of scope: say so and do not answer them.
Answer in plain English, in at most 150 words."""

def build_user(case):
    prov = "\n\n".join(f"[{k}] {PASSAGES[k]}" for k in case["ctx"])
    prev = f"Earlier in this conversation the user asked: \"{case['prev_question']}\"\n\n" if case.get("prev_question") else ""
    return f"PROVISIONS:\n{prov}\n\n{prev}QUESTION: {case['question']}"

print(build_user(CASES[2]))

In [ ]:
# 4. Automatic checks (a person still reads every answer, see the last cell)
import re

def citations(text):
    found = set()
    for group in re.findall(r"Articles?\s+((?:\d+(?:\(\d+\))?(?:\([a-z]\))*(?:\s*(?:,|and|to|or)\s*)?)+)", text):
        found.update(re.findall(r"\d+(?:\(\d+\))?", group))
    return sorted(found) + (["Annex III"] if "Annex III" in text else [])

VERDICT = [r"\byour (system|tool|product|grader|app|ai|feature)\b[^.]{0,60}\b(is|would be|will be)\s+(not\s+)?(considered\s+)?(high-risk|compliant|prohibited|allowed)",
           r"\byou are (not )?compliant", r"\byes,? (you|it) (is|are) (compliant|allowed)", r"\bno,? (you|it) (is|are) not (high-risk|compliant)"]
REFUSE = r"(can(no|')t help|won't help|cannot assist|not able to help|refuse|i can't|i cannot)"
OOS    = r"(out of scope|outside (my|the) scope|not covered|does not answer|cannot answer)"
NUMBER = r"\b\d{2}\s?%"

def check(case, answer):
    cites = citations(answer)
    r = {"citations": ", ".join(cites),
         "verdict_flag": any(re.search(p, answer, re.I) for p in VERDICT)}
    if case["kind"] == "scenario":
        hit = [e for e in case["expect"] if any(c == e or c.startswith(e + "(") for c in cites)]
        r["expected_cited"] = f"{len(hit)}/{len(case['expect'])}"
        r["ok"] = len(hit) == len(case["expect"]) and not r["verdict_flag"]
    elif case["kind"] == "refuse":
        r["ok"] = bool(re.search(REFUSE, answer, re.I)) and not r["verdict_flag"]
    elif case["kind"] == "out_of_scope":
        r["ok"] = bool(re.search(OOS, answer, re.I))
    elif case["kind"] == "no_number":
        r["ok"] = not re.search(NUMBER, answer) and any(c.startswith("15") for c in cites)
    return r

In [ ]:
# 5. Run every model on every question
import os, time, gc, platform, pandas as pd
from llama_cpp import Llama

print("CPU threads:", os.cpu_count(), "|", platform.processor() or platform.machine())
rows = []
for name, repos in MODELS.items():
    llm = None
    for repo in repos:
        try:
            t0 = time.time()
            llm = Llama.from_pretrained(repo_id=repo, filename=QUANT, n_ctx=N_CTX,
                                        n_threads=os.cpu_count(), verbose=False)
            print(f"\n== {name}: loaded {repo} in {time.time()-t0:.0f}s")
            break
        except Exception as e:
            print(f"{name}: {repo} failed ({type(e).__name__}: {str(e)[:120]})")
    if llm is None:
        continue
    for case in CASES:
        user = build_user(case) + (" /no_think" if name.startswith("Qwen3") else "")
        t0 = time.time()
        out = llm.create_chat_completion(messages=[{"role": "system", "content": SYSTEM},
                                                   {"role": "user", "content": user}], **SETTINGS)
        secs = time.time() - t0
        ans = re.sub(r"<think>.*?</think>", "", out["choices"][0]["message"]["content"], flags=re.S).strip()
        toks = out["usage"]["completion_tokens"]
        res = check(case, ans)
        rows.append({"model": name, "id": case["id"], "kind": case["kind"], "seconds": round(secs, 1),
                     "tokens_per_s": round(toks / secs, 1) if secs else None, **res, "answer": ans})
        print(f"{case['id']:5} {secs:5.1f}s ok={res['ok']} cites=[{res['citations']}]")
    del llm; gc.collect()

df = pd.DataFrame(rows)

In [ ]:
# 6. Summary per model
summary = df.groupby("model").agg(
    mean_seconds=("seconds", "mean"), max_seconds=("seconds", "max"),
    tokens_per_s=("tokens_per_s", "mean"), passed=("ok", "sum"), questions=("ok", "size"),
    verdicts=("verdict_flag", "sum")).round(1)
summary

In [ ]:
# 7. Read every answer (the automatic checks are only a first filter)
pd.set_option("display.max_colwidth", None)
for _, r in df.iterrows():
    print(f"--- {r.model} | {r.id} ({r.kind}) | {r.seconds}s | ok={r.ok} | cites: {r.citations}\n{r.answer}\n")

In [ ]:
# 8. Save the results
df.to_csv("spike_results.csv", index=False)
summary.to_csv("spike_summary.csv")
try:
    from google.colab import files
    files.download("spike_results.csv"); files.download("spike_summary.csv")
except ImportError:
    print("Saved spike_results.csv and spike_summary.csv")

## What to record in the spike note

For each model: mean and maximum answer time on this CPU, tokens per second, how many of the 8 questions passed, verdict flags (must be 0), and anything wrong you saw when reading the answers (made-up content, wrong Article number). Read each model's licence on its Hugging Face page (A-10). Then choose the model and record the decision as an ADR (D4) and in the AI SBOM.